# Monitoring notebook — the four layers from the Langfuse traces · ParcelCheck (P4)
Task J · Sessions 6 + 7 · `06_07_experiments_operations.md` §3. Run from the repo root: `uv run jupyter lab evals/monitoring.ipynb`.

- **Live:** reads the last 7 days of `refund-claim` traces and their scores from Langfuse Cloud (keys in `.env`).
- **DEMO:** if Langfuse is not reachable, a synthetic week labelled `DEMO-…` is used to show the layout. **DEMO numbers are never results.**
- Business metrics (leakage € per 1,000, cost per case) come from the weekly human sample, not from the traces.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

root = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
os.chdir(root)
sys.path.insert(0, str(root))
load_dotenv()

In [ ]:
from src.evaluation.monitoring import by_segment, demo_frame, fetch, layer_table, to_frame

DAYS = 7
try:
    traces, scores = fetch(days=DAYS)
    df = to_frame(traces, scores)
    SOURCE = f"LIVE · Langfuse · last {DAYS} days · {len(df)} claims"
    if df.empty:
        raise RuntimeError("no refund-claim trace in the window")
except Exception as error:
    df = demo_frame(200)
    SOURCE = f"DEMO (synthetic, not a result) · reason: {type(error).__name__}: {str(error)[:80]}"
print(SOURCE)
df.head()

## 1 · The rows that page someone (`06_07` §3.1)

In [ ]:
table = layer_table(df)
def fmt(v):
    return f"{v:.1%}" if isinstance(v, float) and 0 <= v <= 1 else (f"{v:.2f}" if isinstance(v, float) else v)
table.assign(value=table["value"].map(fmt))

## 2 · By segment — never the average alone (Simpson, fairness)

In [ ]:
seg = by_segment(df, "auto_resolved").rename(columns={"mean": "L4 share"})
seg["L4 share"] = seg["L4 share"].map(lambda v: f"{v:.1%}")
seg

In [ ]:
blocked = by_segment(df, "input_guardrail_blocked").rename(columns={"mean": "blocked share"})
blocked["blocked share"] = blocked["blocked share"].map(lambda v: f"{v:.1%}")
blocked[blocked["segment"].isin(["language", "amount_band"])]

## 3 · The week, day by day (spikes and drift)

In [ ]:
daily = (df.assign(day=pd.to_datetime(df["timestamp"]).dt.date)
           .groupby("day").agg(claims=("trace_id", "count"), l4_share=("auto_resolved", "mean"),
                               p95_latency_s=("latency_s", lambda s: s.quantile(0.95)),
                               unavailable=("unavailable", "mean")))
daily.round(3)

## 4 · Worst cases to read on Monday (not the averages)

In [ ]:
worst = df[(df["unavailable"]) | (df["input_guardrail_blocked"] == 1) | (df["reviewer_agrees"] == 0)]
worst[["trace_id", "timestamp", "seg_language", "seg_amount_band", "trust_level",
       "input_guardrail_blocked", "reviewer_agrees", "unavailable"]].sort_values("timestamp").head(15)

## What to do with it
- A 🔴 row pages its owner (`06_07` §3.1): product & guardrails → Head of Support · model quality → PM · infrastructure → engineer on call · business → Head of Support + CFO.
- A harmful incident or a broken money threshold → the rollback plan (`06_07` §4.3): switch to suggest-only (L1).
- Paste the live table (not DEMO) into the weekly review with its date.